In [0]:
import json

In [0]:
dbutils.widgets.text("run_date", "")
dbutils.widgets.text("table_config_path", "")
dbutils.widgets.text("ranking_num_shards", "1")

run_date = dbutils.widgets.get("run_date")
table_config_path = dbutils.widgets.get("table_config_path")
ranking_num_shards = int(dbutils.widgets.get("ranking_num_shards") or "1")

print(f"Input parameter run_date:{run_date}")
print(f"Input parameter ranking_num_shards:{ranking_num_shards}")

with open(table_config_path, "r") as file:
    table_config = json.load(file)

db, table_names = table_config["DATABASE"], table_config["TABLE_NAMES"]
target_table = f"{db}.{table_names['ranking_results']}"


def get_shard_table_name(shard_id):
    return f"{db}.{table_names['ranking_results']}_parallel_temp_{int(shard_id)}"


shard_dataframes = []
for shard_id in range(ranking_num_shards):
    shard_table = get_shard_table_name(shard_id)
    if not spark.catalog.tableExists(shard_table):
        raise RuntimeError(f"Expected shard table does not exist: {shard_table}")
    shard_df = spark.table(shard_table).where(f"partition_date = '{run_date}'")
    shard_dataframes.append(shard_df)

if not shard_dataframes:
    raise RuntimeError("No shard outputs were found for merge")

merged_df = shard_dataframes[0]
for shard_df in shard_dataframes[1:]:
    merged_df = merged_df.unionByName(shard_df)

merged_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

for shard_id in range(ranking_num_shards):
    spark.sql(f"DROP TABLE IF EXISTS {get_shard_table_name(shard_id)}")

print(f"Merged {ranking_num_shards} shard outputs into {target_table}")